In [23]:
from astropy.io import fits
import numpy as np
from matplotlib import pyplot as plt
from spectral_cube import SpectralCube
from turbustat.simulator import make_ppv
import astropy.units as u

Data

In [24]:
# Emission line data
line        = "NaN"
lambda_rest = 0
line_id     = "NaN"
line_id2    = "NaN"

# Binning
bins        = 0

# Fake data
region     = 'Synthetic 3D map'
name_reg   = 'Synthetic 3D map'
reg_id     = 'Synthetic 3D map'
inst       = 'NaN'
inst_id    = 'NaN'
type_obj   = 'NaN'

# Physical parameters
pix        = 1  # Instrument resolution [arcsec/pix]
dist       = 1  # Distance [parsecs]
pc         = 1
s0         = 0
noise      = 0


3D density field

In [25]:
with fits.open("exponential_3d_density.fits") as hdul:
    density = hdul[0].data
    header_dens  = hdul[0].header

In [26]:
print(header_dens )

SIMPLE  =                    T / conforms to FITS standard                      BITPIX  =                  -32 / array data type                                NAXIS   =                    3 / number of array dimensions                     NAXIS1  =                  256                                                  NAXIS2  =                  256                                                  NAXIS3  =                  256                                                  MODEL   = 'exponential'                                                         SEED    =             19970221                                                  M3D     =                  1.0                                                  LSCALE_X=                   16                                                  LSCALE_Y=                   16                                                  LSCALE_Z=                   16                                                  ANGLE_X =                  0.8          

In [27]:
model_dens = header_dens["MODEL"]
seed_dens  = header_dens["SEED"]


3D velocity field

In [28]:
with fits.open('exponential_3d_velocity.fits') as hdul:
    velocity = hdul[0].data
    header_vel  = hdul[0].header

In [29]:
model_vel = header_vel ["MODEL"]
seed_vel  = header_vel ["SEED"]
len_scale = header_vel ["LSCALE_X"]
m3D       = header_vel["M3D"]
r0 = len_scale * (np.log(2)**(1 / m3D))
print(len_scale, m3D, r0)

16 1.0 11.090354888959125


In [30]:
print(header_vel)

SIMPLE  =                    T / conforms to FITS standard                      BITPIX  =                  -32 / array data type                                NAXIS   =                    3 / number of array dimensions                     NAXIS1  =                  256                                                  NAXIS2  =                  256                                                  NAXIS3  =                  256                                                  MODEL   = 'exponential'                                                         SEED    =             19880127                                                  M3D     =                  1.0                                                  LSCALE_X=                   16                                                  LSCALE_Y=                   16                                                  LSCALE_Z=                   16                                                  ANGLE_X =                  0.8          

Spectral Cube

In [31]:
density.var()

0.98615587

In [32]:
density = (density-density.mean())/density.std()

In [33]:
density.var()

0.9999986

In [37]:
for n in [0, 1, 2]:
    eps = np.exp(n * density)

    print(
        n,
        "mean(z) =", density.mean(),
        "var(z) =", density.var(),
        "std(log eps) =", np.log(eps).std(),
        "var(log eps) =", np.log(eps).var(),
        "mean(eps) =", eps.mean(),
        "var(eps) =", eps.var(),
    )

0 mean(z) = 5.2677933e-09 var(z) = 0.9999986 std(log eps) = 0.0 var(log eps) = 0.0 mean(eps) = 1.0 var(eps) = 0.0
1 mean(z) = 5.2677933e-09 var(z) = 0.9999986 std(log eps) = 0.9999992 var(log eps) = 0.9999985 mean(eps) = 1.6532074 var(eps) = 4.7428164
2 mean(z) = 5.2677933e-09 var(z) = 0.9999986 std(log eps) = 1.9999986 var(log eps) = 3.9999945 mean(eps) = 7.475926 var(eps) = 2875.9055


In [34]:
velocity = velocity* u.km / u.s   

velocity_64 = u.Quantity(
    np.ascontiguousarray(velocity.value, dtype=np.float64),
    unit=velocity.unit,
)

In [11]:
for sig_E in [0, 1, 2]:

    print(f"\nRunning sig_E = {sig_E}")

    # -------------------------------------------------------
    # Emissivity
    # -------------------------------------------------------
    emissivity = np.exp(sig_E * density) * u.cm**-3

    emissivity_64 = u.Quantity(
        np.ascontiguousarray(emissivity.value, dtype=np.float64),
        unit=emissivity.unit,
    )

    # -------------------------------------------------------
    # PPV cube
    # -------------------------------------------------------
    cube_hdu = make_ppv(
        velocity_64,
        emissivity_64,
        los_axis=0,
        T=10000 * u.K,
        chan_width=0.5 * u.km / u.s,
        v_min=-20 * u.km / u.s,
        v_max=20 * u.km / u.s,
    )

    cube = SpectralCube.read(cube_hdu)

    # -------------------------------------------------------
    # Moments
    # -------------------------------------------------------
    sb = cube.moment0().value
    vv = cube.moment1().value

    sig = np.std(vv)
    sig2 = np.var(vv)

    N = np.sqrt(vv.shape[0] * vv.shape[1]) * pix * pc * (2**bins)

    # -------------------------------------------------------
    # Filename
    # -------------------------------------------------------
    name_file = (
        f"gstools_finite_sigE{sig_E}_"
        f"{model_vel}_r{int(r0)}_N{int(N)}_bin{bins}"
    )

    name_id = name_file

    print(name_file)

    # -------------------------------------------------------
    # Plot
    # -------------------------------------------------------
    fig, (ax, axx) = plt.subplots(
        1,
        2,
        sharey=True,
        figsize=(10, 4),
    )

    im = ax.imshow(sb, origin="lower", cmap="magma")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("(-)", rotation=270, labelpad=15)

    im = axx.imshow(vv, origin="lower", cmap="RdBu_r")
    cbar = fig.colorbar(im, ax=axx)
    cbar.set_label("km/s", rotation=270, labelpad=15)

    ax.set_xlabel("pix")
    ax.set_ylabel("pix")
    axx.set_xlabel("pix")

    ax.set_title("Moment 0")
    axx.set_title("Moment 1")

    fig.suptitle(name_file, fontsize=12)

    fig.savefig(
        name_file + ".png",
        dpi=300,
        bbox_inches="tight",
    )

    # Important when creating several plots in a loop
    plt.close(fig)

    # -------------------------------------------------------
    # FITS primary header
    # -------------------------------------------------------
    primary = fits.PrimaryHDU()

    primary.header["FILE"]     = (name_file, "Fits file name")
    primary.header["NAME_ID"]  = (name_id, "pipeline identifier")
    primary.header["REG"]      = (region, "Region catalog name")
    primary.header["NAME_REG"] = (name_reg, "Region name")
    primary.header["REG_ID"]   = (reg_id, "Region identifier")
    primary.header["TYPE_OBJ"] = (type_obj, "Type object")
    primary.header["DIST"]     = (dist, "Distance in parsecs")
    primary.header["INSTR"]    = (inst, "Instrument")
    primary.header["LINE"]     = (line, "Emission line name")
    primary.header["LINE_ID"]  = (line_id, "Emission line identifier")
    primary.header["L_REST"]   = (lambda_rest, "Emission line name")

    primary.header["SIG"]      = (sig, "Standard deviation vel map")
    primary.header["SIG2"]     = (sig2, "Vel Variance map")

    primary.header["S0"]       = (s0, "Atmospheric seeing")
    primary.header["BOX_SIZE"] = (N, "Observational box_size")
    primary.header["PC"]       = (pc, "parsec convertion")
    primary.header["PIX"]      = (pix * (2**bins), "Instrument pixel scale")
    primary.header["BINSIZE"]  = (bins, "Spatial binning factor")
    primary.header["noise"]    = (noise, "Instrumental noise")

    primary.header["M3D"]      = (m3D, "alpha model")
    primary.header["SEED1"]    = (seed_dens, "Random seed surface brightness")
    primary.header["SEED2"]    = (seed_vel, "Random seed velocity")
    primary.header["MODEL"]    = (model_vel, "GSTools model")
    primary.header["L_SCALE"]  = (len_scale, "GSTools length scale")

    primary.header["r0"]       = (r0, "Correlation length")
    primary.header["sigE"]     = (sig_E, "Emissivity sigma")

    primary.header["AUTHOR"]   = ("J. Garcia-Vazquez", "File creator")

    # -------------------------------------------------------
    # Moment extensions
    # -------------------------------------------------------
    hdu_m0 = fits.ImageHDU(data=sb, name="MOM0")
    hdu_m1 = fits.ImageHDU(data=vv, name="MOM1")

    # -------------------------------------------------------
    # Write FITS
    # -------------------------------------------------------
    hdul = fits.HDUList([primary, hdu_m0, hdu_m1])

    hdul.writeto(
        name_id + ".fits",
        overwrite=True,
    )

    print(f"Saved: {name_id}.fits")


Running sig_E = 0
gstools_finite_sigE0_exponential_r11_N256_bin0
Saved: gstools_finite_sigE0_exponential_r11_N256_bin0.fits

Running sig_E = 1
gstools_finite_sigE1_exponential_r11_N256_bin0
Saved: gstools_finite_sigE1_exponential_r11_N256_bin0.fits

Running sig_E = 2
gstools_finite_sigE2_exponential_r11_N256_bin0
Saved: gstools_finite_sigE2_exponential_r11_N256_bin0.fits
